# 🎓 Data Science Mini Project: Online Course Completion Analysis

**Author:** Data Science Student  
**Project Type:** Exploratory Data Analysis (EDA) & Educational Analytics  
**Technologies:** Python, Pandas, NumPy, Matplotlib, Seaborn  

---

## 📌 1. Project Overview & Problem Statement

Online education platforms (such as Coursera, edX, and Udemy) face a well-documented industry challenge: **high enrollment rates but low course completion rates** (often below 20-30%).

### Objective:
1. Analyze student enrollments across various courses and domains.
2. Evaluate the number of modules completed by learners.
3. Calculate individual and aggregate course completion percentages.
4. Compare completion metrics across diverse course categories (Programming, Data Science, Web Dev, Business & Marketing).
5. Identify highest and lowest performing courses.
6. Provide actionable recommendations to improve student retention and engagement.


## 🛠️ Step 1: Importing Required Libraries
We begin by importing the core data science libraries in Python:
- **Pandas:** For data manipulation and tabular analysis.
- **NumPy:** For numerical operations and vectorized math.
- **Matplotlib & Seaborn:** For visual storytelling and professional charts.


In [ ]:
# Step 1: Import Essential Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configure plotting aesthetics and warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print('All libraries successfully loaded!')


## 📊 Step 2: Loading the Dataset
We load the online course learning dataset `online_course_data.csv`.  
*(Note: This dataset contains realistic synthetic records of 600 student enrollments across 8 courses.)*


In [ ]:
# Step 2: Load the Dataset
df_raw = pd.read_csv('online_course_data.csv')

# Display first 5 records
print('Dataset Shape (Rows, Columns):', df_raw.shape)
df_raw.head()


## 🔍 Step 3: Data Inspection & Preprocessing
Before analyzing data, a data scientist must verify:
1. Data types and column information (`.info()`)
2. Duplicate records (`.duplicated()`)
3. Missing or null values (`.isnull()`)


In [ ]:
# Step 3.1: Check column data types and non-null counts
df_raw.info()


In [ ]:
# Step 3.2: Remove duplicate rows
duplicate_count = df_raw.duplicated().sum()
print(f'Total duplicate rows detected: {duplicate_count}')

df = df_raw.drop_duplicates().copy()
print(f'Rows after removing duplicates: {len(df)}')


In [ ]:
# Step 3.3: Inspect missing values
print('Missing values per column before cleaning:')
print(df.isnull().sum())

# Handle missing values accurately:
# If Completed Modules is missing, impute with 0
df['Completed Modules'] = df['Completed Modules'].fillna(0).astype(int)

# Recalculate Completion Percentage accurately: (Completed Modules / Total Modules) * 100
df['Completion Percentage'] = ((df['Completed Modules'] / df['Total Modules']) * 100).round(2)

# Correct Completion Status: 'Completed' if Completed Modules == Total Modules else 'Incomplete'
df['Completion Status'] = np.where(df['Completed Modules'] == df['Total Modules'], 'Completed', 'Incomplete')

# Convert Enrollment Date to datetime object
df['Enrollment Date'] = pd.to_datetime(df['Enrollment Date'])

print('\nCleaning complete! Remaining missing values:', df.isnull().sum().sum())


## 📈 Step 4: Descriptive Statistics & Global Metrics
Let us observe summary statistics of the numeric features to understand the distribution of modules and completion progress.


In [ ]:
# Summary statistics of numeric attributes
df.describe().round(2)


In [ ]:
# Global platform performance metrics
total_students = len(df)
total_completed = (df['Completion Status'] == 'Completed').sum()
total_incomplete = (df['Completion Status'] == 'Incomplete').sum()
global_completion_rate = (total_completed / total_students) * 100
avg_progress = df['Completion Percentage'].mean()

print(f'Total Student Enrollments: {total_students}')
print(f'Total Courses Completed: {total_completed} ({global_completion_rate:.2f}%)')
print(f'Total Courses Incomplete: {total_incomplete} ({(100 - global_completion_rate):.2f}%)')
print(f'Average Student Progress: {avg_progress:.2f}%')


## 🎯 Step 5: Course-Wise & Category-Wise Analysis
In this step, we aggregate data by **Course Name** and **Course Category** to identify:
- Total enrollments per course
- Course completion count and percentage
- Average modules completed by enrolled students


In [ ]:
# 5.1: Course-wise aggregation
course_perf = df.groupby(['Course Name', 'Course Category']).agg(
    Total_Enrollments=('Student ID', 'count'),
    Total_Modules=('Total Modules', 'first'),
    Avg_Completed_Modules=('Completed Modules', 'mean'),
    Completed_Count=('Completion Status', lambda x: (x == 'Completed').sum()),
    Incomplete_Count=('Completion Status', lambda x: (x == 'Incomplete').sum()),
    Avg_Student_Pct=('Completion Percentage', 'mean')
).reset_index()

course_perf['Completion_Rate_%'] = ((course_perf['Completed_Count'] / course_perf['Total_Enrollments']) * 100).round(2)
course_perf['Avg_Completed_Modules'] = course_perf['Avg_Completed_Modules'].round(2)
course_perf['Avg_Student_Pct'] = course_perf['Avg_Student_Pct'].round(2)

# Sort courses by Completion Rate descending
course_perf = course_perf.sort_values(by='Completion_Rate_%', ascending=False)
course_perf


In [ ]:
# 5.2: Category-wise aggregation
cat_perf = df.groupby('Course Category').agg(
    Total_Enrollments=('Student ID', 'count'),
    Completed_Count=('Completion Status', lambda x: (x == 'Completed').sum()),
    Incomplete_Count=('Completion Status', lambda x: (x == 'Incomplete').sum()),
    Avg_Student_Pct=('Completion Percentage', 'mean')
).reset_index()

cat_perf['Category_Completion_Rate_%'] = ((cat_perf['Completed_Count'] / cat_perf['Total_Enrollments']) * 100).round(2)
cat_perf['Avg_Student_Pct'] = cat_perf['Avg_Student_Pct'].round(2)
cat_perf = cat_perf.sort_values(by='Category_Completion_Rate_%', ascending=False)
cat_perf


In [ ]:
# 5.3: Identify Top & Lowest Performing Courses
top_course = course_perf.iloc[0]
bottom_course = course_perf.iloc[-1]

print(f'🏆 Highest Completion Rate: "{top_course["Course Name"]}" with {top_course["Completion_Rate_%"]}%')
print(f'⚠️ Lowest Completion Rate: "{bottom_course["Course Name"]}" with {bottom_course["Completion_Rate_%"]}%')


## 📊 Step 6: Data Visualizations
We now generate the requested charts to visually communicate patterns in enrollment, completion rates, category differences, and course difficulty.


In [ ]:
# Visualization 1: Student Enrollment by Course
plt.figure(figsize=(10, 6))
order = df['Course Name'].value_counts().index
ax = sns.countplot(data=df, y='Course Name', order=order, hue='Course Name', palette='viridis', legend=False)
plt.title('Total Student Enrollment by Course', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Number of Enrolled Students', fontsize=11)
plt.ylabel('Course Name', fontsize=11)

# Add data labels
for p in ax.patches:
    w = p.get_width()
    ax.annotate(f'{int(w)}', (w + 1.5, p.get_y() + p.get_height() / 2),
                ha='left', va='center', fontweight='bold', fontsize=10)

plt.xlim(0, max(df['Course Name'].value_counts()) * 1.15)
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 2: Course-Wise Completion Percentage Bar Chart
plt.figure(figsize=(11, 6))
sorted_c = course_perf.sort_values(by='Completion_Rate_%', ascending=True)
colors = ['#e74c3c' if r < 50 else '#f39c12' if r < 65 else '#27ae60' for r in sorted_c['Completion_Rate_%']]

bars = plt.barh(sorted_c['Course Name'], sorted_c['Completion_Rate_%'], color=colors, edgecolor='black', height=0.6)
plt.axvline(x=global_completion_rate, color='blue', linestyle='--', linewidth=1.5, label=f'Platform Avg ({global_completion_rate:.1f}%)')

for bar in bars:
    w = bar.get_width()
    plt.text(w + 1.2, bar.get_y() + bar.get_height() / 2, f'{w:.1f}%', ha='left', va='center', fontweight='bold', fontsize=10)

plt.title('Course-wise Completion Rates (%)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Completion Rate (%)', fontsize=11)
plt.ylabel('Course Name', fontsize=11)
plt.xlim(0, 100)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 3: Pie Chart of Completed vs Incomplete Courses
plt.figure(figsize=(7, 7))
status_counts = df['Completion Status'].value_counts()

plt.pie(
    status_counts, 
    labels=status_counts.index, 
    autopct='%1.1f%%', 
    startangle=140,
    colors=['#2ecc71', '#e74c3c'], 
    explode=(0.05, 0), 
    shadow=True,
    textprops={'fontsize': 13, 'fontweight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
plt.title('Overall Course Completion Status Distribution', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 4: Category-Wise Completion Rates Bar Chart
plt.figure(figsize=(9, 5))
ax4 = sns.barplot(data=cat_perf, x='Course Category', y='Category_Completion_Rate_%', hue='Course Category', palette='magma', legend=False, edgecolor='black')
plt.title('Course Completion Rate by Category (%)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Course Category', fontsize=11)
plt.ylabel('Completion Rate (%)', fontsize=11)
plt.ylim(0, 100)

for p in ax4.patches:
    h = p.get_height()
    ax4.annotate(f'{h:.1f}%', (p.get_x() + p.get_width() / 2, h + 2),
                 ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Visualization 5: Performance Heatmap across Courses
plt.figure(figsize=(10, 6))
hm_data = course_perf.set_index('Course Name')[
    ['Total_Enrollments', 'Total_Modules', 'Avg_Completed_Modules', 'Avg_Student_Pct', 'Completion_Rate_%']
].rename(columns={
    'Total_Enrollments': 'Enrollments',
    'Total_Modules': 'Total Modules',
    'Avg_Completed_Modules': 'Avg Done Modules',
    'Avg_Student_Pct': 'Avg Student Progress %',
    'Completion_Rate_%': 'Completion Rate %'
})

sns.heatmap(hm_data, annot=True, fmt='.1f', cmap='YlGnBu', cbar=True, linewidths=1, linecolor='gray')
plt.title('Course Performance Heatmap (Key Metrics Comparison)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Performance Metrics', fontsize=11)
plt.ylabel('Course Name', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 6: Box Plot of Completed Modules across Categories
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Course Category', y='Completed Modules', hue='Course Category', palette='Set2', legend=False)
plt.title('Distribution of Modules Completed across Categories', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Course Category', fontsize=11)
plt.ylabel('Number of Completed Modules', fontsize=11)
plt.tight_layout()
plt.show()


## 📝 Step 7: Key Findings & Actionable Recommendations

### Summary of Findings:
1. **Most Popular Course:** `Python for Beginners` had the highest enrollment (122 students), demonstrating strong market demand for introductory coding skills.
2. **Highest Completion Rate:** `Digital Marketing Strategies` achieved an **80.30% completion rate**, benefiting from a concise curriculum (8 modules) and practical project assignments.
3. **Lowest Completion Rate:** `Machine Learning & Deep Learning` registered the lowest completion rate at **30.16%**, accompanied by the highest module count (16 modules) and steep mathematical prerequisites.
4. **Category Disparity:** `Business & Marketing` courses led the platform with a **75.00% category completion rate**, whereas `Data Science` had the lowest at **39.47%**.

### Recommendations for the Learning Platform:
- **Micro-Learning Architecture:** Break down intensive 14–16 module courses (like Machine Learning) into bite-sized 4–6 module sub-certifications.
- **Early Intervention Triggers:** Identify students who stall before Module 4 and send automated reminder nudges or offer mentor Q&A sessions.
- **Interactive Coding Sandboxes:** Add hands-on coding exercises inside the browser to reduce friction in technical courses.
